# OMIP Technical Preprocessing: The Data Quality Gate
## 1. Objectives and Modular Strategy

Before proceeding to feature engineering or model training, we must ensure the **technical integrity** of the raw market data. Following the project's modular architecture, this notebook acts as a validation layer for the technical cleaning script.

**Technical Cleaning Goals:**
* **Schema Standardization:** Renaming columns to a consistent snake_case or specific project naming conventions.
* **Type Coercion:** Ensuring prices and Open Interest are stored as floats, and dates as datetime objects.
* **Duplicate Elimination:** Removing any overlapping data points from multiple scraper runs, keeping only the most recent (last) entry.
* **Integrity Validation:** Enforcing rules (e.g., Spot prices cannot be null) to prevent pipeline crashes.

The heavy lifting is performed by `src/data/clean_omip.py`, ensuring that cleaning logic is reusable across different environments.

In [7]:
import pandas as pd
import sys
import os
from pathlib import Path

sys.path.append(os.path.abspath(r'C:\Users\Alejandro\GitHub\Data-Driven-Strategies-for-Financial-Resilience-in-Energy-Procurement'))

from src.data.clean_omip import clean_omip_dataframe
from src.config.constants import DATE_COLUMN, SPOT_PRICE_COLUMN

# Define data paths
raw_data_path = Path(r"C:\Users\Alejandro\GitHub\Data-Driven-Strategies-for-Financial-Resilience-in-Energy-Procurement\data\raw\omip\omip_prices_raw.csv")
interim_output_path = Path(r"C:\Users\Alejandro\GitHub\Data-Driven-Strategies-for-Financial-Resilience-in-Energy-Procurement\data\interim\omip_clean.csv")

print("✅ Modular cleaning tools imported successfully.")

✅ Modular cleaning tools imported successfully.


## 2. Pipeline Execution

We load the raw CSV and pass it through the `clean_omip_dataframe` function. This allows us to inspect the data in-memory before saving the interim version.

In [8]:
# Load raw data
df_raw = pd.read_csv(raw_data_path)

# Execute the cleaning logic defined in src/data/clean_omip.py
df_clean = clean_omip_dataframe(df_raw)

# Save the interim (cleaned) version
df_clean.to_csv(interim_output_path, index=False)

print(f"✅ Technical cleaning complete.")
print(f"Raw Shape: {df_raw.shape} | Cleaned Shape: {df_clean.shape}")

✅ Technical cleaning complete.
Raw Shape: (2192, 14) | Cleaned Shape: (2192, 14)


## 3. Data Quality Audit

To guarantee the quality of the "Data Quality Gate", we perform a quick check on missing values and data types. This ensures that the downstream Feature Engineering or Merge steps won't face unexpected 'Object' types or Nulls.

In [9]:
# Check for nulls in the cleaned dataset
null_report = df_clean.isnull().sum().to_frame(name='Missing Values')
null_report['Dtype'] = df_clean.dtypes

print("Data Quality Report (Interim Dataset):")
display(null_report)

# Verify chronological order
is_sorted = df_clean[DATE_COLUMN].is_monotonic_increasing
print(f"\nChronological Integrity: {'Correct (Sorted)' if is_sorted else 'Incorrect (Unsorted!)'}")

Data Quality Report (Interim Dataset):


,Missing Values,Dtype
date,0,datetime64[us]
Spot_Price_SPEL,0,float64
Future_M1_Price,1,float64
Future_M1_OpenInterest,1,float64
Future_M2_Price,1,float64
Future_M2_OpenInterest,1,float64
Future_M3_Price,1,float64
Future_M3_OpenInterest,1,float64
Future_M4_Price,1,float64
Future_M4_OpenInterest,1,float64



Chronological Integrity: Correct (Sorted)


## 4. Anomaly Resolution & Handoff to Feature Engineering

**Data Quality Finding: The "1 Missing Value" Anomaly**
The audit reveals exactly 0 missing values for the Spot Market, but exactly 1 missing value across all Futures and Open Interest columns. 

**Business Explanation:**
This is not a data corruption issue, but a structural market reality. The dataset begins on January 1st, 2020. 
* The **Spot Market (OMIE)** operates 365 days a year, hence no missing values.
* The **Futures Market (OMIP)** is a financial exchange that closes on public holidays (such as New Year's Day).

**Architectural Decision:**
We intentionally do **not** impute this value during the technical cleaning phase (`clean_omip.py`), as technical scripts should not alter financial history. Instead, this anomaly is handled via a backward fill (`bfill`) strategy within the **Feature Engineering** phase (`build_feature_matrix.py`), ensuring our Reinforcement Learning agent has a complete initial state before calculating rolling windows and lags.

**Conclusion:**
The dataset `omip_clean.csv` has passed the technical quality gate and is certified ready for Analytical Feature Engineering.